# 1. Sử dụng kỹ thuật OCR để trích xuất đánh giá người dùng từ dữ liệu hình ảnh.

In [1]:
import os
from dotenv import load_dotenv

# Tải các biến môi trường từ file .env lên hệ thống
load_dotenv() 

# Bây giờ thư viện genai.Client() sẽ tự động tìm thấy key
from google import genai
client = genai.Client() 
# (Lưu ý: với SDK mới, nếu có biến GEMINI_API_KEY trong môi trường, bạn không cần truyền api_key=... vào Client() nữa)

In [2]:
import os
import json
import glob
import asyncio
import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from tqdm.asyncio import tqdm # THÊM TQDM ĐỂ HIỂN THỊ THANH TIẾN TRÌNH

# 1. Khởi tạo Client
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# 2. Định nghĩa Pydantic Schema
class Review(BaseModel):
    review_text: str = Field(description="Chỉ chứa phần nội dung văn bản đánh giá cốt lõi của khách hàng.")

class ReviewExtraction(BaseModel):
    reviews: list[Review]

# 3. Hàm xử lý 1 ảnh với cơ chế tự động thử lại (Retry)
async def process_image_async(image_path: str, semaphore: asyncio.Semaphore, max_retries: int = 3):
    async with semaphore:
        for attempt in range(max_retries):
            try:
                image = Image.open(image_path)
                
                prompt = """
                Hãy đọc bức ảnh và trích xuất nội dung các câu đánh giá. 
                Tuyệt đối CHỈ lấy phần văn bản nhận xét của khách hàng. 
                BỎ QUA TẤT CẢ các thông tin sau: tên user, số sao, ngày tháng, phân loại hàng, kích cỡ, màu sắc trên nhãn.
                """
                
                response = await client.aio.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=[image, prompt],
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=ReviewExtraction,
                        temperature=0.1,
                        max_output_tokens=8192 
                    )
                )
                
                return json.loads(response.text)

            except json.JSONDecodeError as e:
                # Thay print bằng tqdm.write để không làm vỡ thanh tiến trình
                tqdm.write(f"[!] Lỗi cấu trúc JSON tại {image_path} (Thử lại {attempt+1}/{max_retries}): {e}")
                await asyncio.sleep(1) 
                
            except Exception as e:
                error_msg = str(e)
                if "503" in error_msg or "429" in error_msg:
                    wait_time = 2 ** attempt 
                    tqdm.write(f"[!] Server bận (503/429) tại {image_path}. Đợi {wait_time}s rồi thử lại...")
                    await asyncio.sleep(wait_time)
                else:
                    tqdm.write(f"[X] Lỗi nghiêm trọng tại {image_path}: {e}")
                    break 
                    
        return None 

# 4. Hàm quản lý Batch Async với Checkpoint
async def batch_extract_reviews_to_excel(image_folder: str, output_file: str, max_concurrent: int = 10, checkpoint_step: int = 100):
    image_paths = glob.glob(os.path.join(image_folder, "*.[pj][pn]*[g]"))
    if not image_paths:
        print("Không tìm thấy ảnh nào trong thư mục.")
        return

    total_images = len(image_paths)
    print(f"Bắt đầu xử lý {total_images} ảnh với {max_concurrent} luồng song song...")
    
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [process_image_async(path, semaphore) for path in image_paths]
    
    all_reviews = []
    failed_count = 0
    processed_count = 0
    
    # Khởi tạo thanh tiến trình TQDM
    pbar = tqdm(total=total_images, desc="Tiến độ trích xuất", unit="ảnh")
    
    # Sử dụng as_completed để xử lý ngay khi có task hoàn thành
    for coro in asyncio.as_completed(tasks):
        res = await coro
        processed_count += 1
        pbar.update(1) # Cập nhật thanh tiến trình thêm 1 ảnh
        
        # Thêm kết quả vào list tổng
        if res and 'reviews' in res:
            for review in res['reviews']:
                all_reviews.append({"Nội dung đánh giá": review['review_text']})
        else:
            failed_count += 1
            
        # LƯU CHECKPOINT SAU MỖI 'checkpoint_step' (MẶC ĐỊNH LÀ 100 ẢNH)
        if processed_count % checkpoint_step == 0:
            if all_reviews:
                df = pd.DataFrame(all_reviews)
                df.to_excel(output_file, index=False, engine='openpyxl')
                tqdm.write(f" 💾 [Checkpoint] Đã lưu tạm {len(all_reviews)} đánh giá vào {output_file} (Xử lý {processed_count}/{total_images} ảnh).")

    pbar.close() # Đóng thanh tiến trình
            
    # Ghi ra file Excel lần cuối (đảm bảo lưu những ảnh lẻ còn lại)
    if all_reviews:
        df = pd.DataFrame(all_reviews)
        df.to_excel(output_file, index=False, engine='openpyxl')
        print(f"\n✅ Hoàn tất toàn bộ! Đã trích xuất tổng cộng {len(all_reviews)} đánh giá.")
        print(f"⚠️ Có {failed_count} ảnh không thể xử lý sau nhiều lần thử.")
        print(f"📁 Dữ liệu được lưu an toàn tại: {output_file}")
    else:
        print("\n❌ Không có dữ liệu hợp lệ nào được trích xuất.")

# --- Khối chạy thử ---
if __name__ == "__main__":
    INPUT_FOLDER = "./DuLieuAnh" 
    OUTPUT_EXCEL = "raw_fashion_reviews.xlsx" 
    
    # Nếu chưa có thư viện tqdm, bạn cần cài đặt: pip install tqdm
    # Mặc định checkpoint_step=100, hệ thống sẽ lưu file sau mỗi 100 ảnh hoàn thành
    await batch_extract_reviews_to_excel(
        image_folder=INPUT_FOLDER, 
        output_file=OUTPUT_EXCEL, 
        max_concurrent=10, 
        checkpoint_step=100
    )

Bắt đầu xử lý 32 ảnh với 10 luồng song song...


Tiến độ trích xuất: 100%|█████████████████████████████████████████████████████████████| 32/32 [00:27<00:00,  1.15ảnh/s]



✅ Hoàn tất toàn bộ! Đã trích xuất tổng cộng 184 đánh giá.
⚠️ Có 0 ảnh không thể xử lý sau nhiều lần thử.
📁 Dữ liệu được lưu an toàn tại: raw_fashion_reviews.xlsx


# 2. Tiền xử lý văn bản

In [3]:
import os
import re
import json
import asyncio
import pandas as pd
import unicodedata
from langdetect import detect, DetectorFactory
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
from tqdm.asyncio import tqdm

# Đảm bảo kết quả nhận diện ngôn ngữ ổn định
DetectorFactory.seed = 0

# --- 1. CẤU HÌNH API ---
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# --- 2. ĐỊNH NGHĨA SCHEMA CHO GEMINI ---
class CleanedReview(BaseModel):
    id: int = Field(description="ID của đánh giá tương ứng trong mảng đầu vào.")
    is_valid_fashion_review: bool = Field(description="True nếu hợp lệ, False nếu là spam/văn mẫu.")
    cleaned_text: str = Field(description="Đánh giá đã được chuẩn hóa. Để trống nếu không hợp lệ.")

class BatchCleanedReviews(BaseModel):
    results: list[CleanedReview]

# --- 3. BỘ CÔNG CỤ TIỀN XỬ LÝ VĂN BẢN (TỪ JUPYTER NOTEBOOK) ---
abbreviation_dict = {
    r'\bdc\b': 'được', r'\bđc\b': 'được', r'\bđk\b': 'được',
    r'\bko\b': 'không', r'\bk\b': 'không', r'\bkg\b': 'không', r'\bhk\b': 'không', r'\bhong\b': 'không', r'\bkhg\b': 'không',
    r'\bsp\b': 'sản phẩm', r'\bng\b': 'người', r'\bmn\b': 'mọi người', r'\bmng\b': 'mọi người',
    r'\bh\b': 'giờ', r'\bkh\b': 'khách', r'\bgw\b': 'giao', r'\bnt\b': 'nhắn tin',
    r'\bshop\b': 'cửa hàng', r'\bsop\b': 'cửa hàng', r'\bshops\b': 'cửa hàng',
    r'\btks\b': 'cảm ơn', r'\bthanks\b': 'cảm ơn', r'\bthank\b': 'cảm ơn',
    r'\bck\b': 'chồng', r'\bbt\b': 'bình thường', r'\bmik\b': 'mình',
    r'\blm\b': 'làm', r'\bcskh\b': 'chăm sóc khách hàng', r'\btphcm\b': 'thành phố hồ chí minh',
    r'\bvãi\b': 'vải', r'\btrc\b': 'trước', r'\bnv\b': 'nhân viên',
    r'\bvs\b': 'với', r'\bfrom\b': 'form', r'\bfom\b': 'form', r'\bphom\b': 'form',
    r'\bsezi\b': 'size', r'\bqc\b': 'quảng cáo', r'\bcx\b': 'cũng', r'\bib\b': 'nhắn tin'
}

def is_vietnamese(text):
    """Kiểm tra xem câu có phải tiếng Việt hay không"""
    if not text or len(text) < 5:
        return True 
    try:
        return detect(text) != 'en'
    except:
        return True

def clean_and_normalize(text):
    if pd.isna(text) or str(text).strip() == "":
        return None
    
    text = str(text)
    
    # 1. Xóa các cụm thuộc tính liệt kê vô nghĩa do hệ thống chèn vào (Kết hợp logic cũ)
    text = re.sub(r"(Màu sắc|Chất liệu|Kích cỡ|Phân loại hàng|Đúng với mô tả):\s*[^,.\n]+[,.\n]?\s*", "", text, flags=re.IGNORECASE)
    
    # 2. Xử lý xuống dòng và Unicode (RẤT QUAN TRỌNG ĐỂ CHỐNG LỖI JSON)
    text = re.sub(r'[\n\r\t]', ' ', text)
    text = unicodedata.normalize('NFC', text).lower()
    
    # 3. Xóa các câu thuần tiếng Anh
    if not is_vietnamese(text):
        return None

    # 4. Chuyển đổi từ viết tắt
    for abbr, full in abbreviation_dict.items():
        text = re.sub(abbr, full, text)
        
    # 5. Làm sạch ký tự đặc biệt và ký tự lặp
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ.,!]', ' ', text)
    
    # 6. Xóa từ rác quá dài (gibberish)
    words = [w for w in text.split() if len(w) <= 20]
    text = " ".join(words)
    
    final_text = re.sub(r'\s+', ' ', text).strip()
    return final_text if final_text != "" else None


# --- 4. GIAI ĐOẠN 1: BỘ LỌC THÔ (TÍCH HỢP) ---
def stage1_basic_cleaning(df: pd.DataFrame, text_col="Nội dung đánh giá") -> pd.DataFrame:
    print(f"[*] Giai đoạn 1: Bắt đầu dọn dẹp cơ bản với {len(df)} dòng dữ liệu...")
    
    df = df.copy()
    
    # Áp dụng hàm làm sạch và chuẩn hóa (Jupyter logic)
    df[text_col] = df[text_col].apply(clean_and_normalize)
    
    # Xóa rỗng (các câu bị đánh giá là None do không phải tiếng Việt hoặc rỗng)
    df = df.dropna(subset=[text_col])
    
    # Quét sạch "văn mẫu" và rác bằng Regex (Pipeline logic)
    spam_patterns = r"hình ảnh.*tính chất minh hoạ|chỉ.*nhận xu|kiếm xu|ảnh.*minh họa|không liên quan|để lấy xu|mang tính chất nhận xu"
    df = df[~df[text_col].str.contains(spam_patterns, flags=re.IGNORECASE, na=False)]
    
    # Xóa trùng lặp
    df = df.drop_duplicates(subset=[text_col])
    
    # Lọc theo độ dài (Chỉ lấy câu có từ 10 đến 100 từ)
    df = df[df[text_col].apply(lambda x: 9 < len(str(x).split()) < 100)]
    
    print(f"[*] Giai đoạn 1 hoàn tất. Còn lại {len(df)} dòng hợp lệ.")
    return df


# --- 5. GIAI ĐOẠN 2: BỘ LỌC TINH BẰNG GEMINI ASYNC ---
async def process_batch_async(batch_data: list[dict], semaphore: asyncio.Semaphore, max_retries: int = 3):
    async with semaphore:
        first_id = batch_data[0]['id'] if batch_data else "Unknown"
        
        for attempt in range(max_retries):
            try:
                input_json = json.dumps(batch_data, ensure_ascii=False)
                prompt = f"""
                Bạn là chuyên gia ngôn ngữ học tiếng Việt. Nhiệm vụ của bạn là kiểm tra lô dữ liệu đánh giá thời trang sau.
                Dữ liệu đã được tiền xử lý cơ bản, hãy xác định đối tượng nào là đánh giá mua hàng hợp lệ (không phải spam, không lạc đề).
                Nếu hợp lệ, hãy trau chuốt lại câu từ cho tự nhiên nhất (giữ nguyên cảm xúc và khía cạnh).
                
                LƯU Ý QUAN TRỌNG: 
                - Bạn phải phản hồi ĐÚNG định dạng JSON. 
                - Mọi dấu ngoặc kép (") bên trong văn bản đánh giá phải được escape cẩn thận thành (\").
                
                Dữ liệu đầu vào:
                {input_json}
                """
                
                response = await client.aio.models.generate_content(
                    model='gemini-3-flash-preview',
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=BatchCleanedReviews,
                        temperature=0.1,
                        max_output_tokens=8192 
                    )
                )
                
                result = json.loads(response.text)
                return result.get("results", [])

            except json.JSONDecodeError as e:
                tqdm.write(f"[!] Lỗi giải mã JSON ở lô ID {first_id} (Thử lại {attempt+1}/{max_retries}): {e}")
                await asyncio.sleep(2)
                
            except Exception as e:
                error_msg = str(e)
                if "503" in error_msg or "429" in error_msg:
                    wait_time = 2 ** attempt
                    tqdm.write(f"[!] Server bận (503/429) ở lô ID {first_id}. Đợi {wait_time}s rồi thử lại...")
                    await asyncio.sleep(wait_time)
                else:
                    tqdm.write(f"[X] Lỗi API nghiêm trọng ở lô ID {first_id}: {error_msg}")
                    return []
                    
        tqdm.write(f"[X] Bỏ qua lô ID {first_id} sau {max_retries} lần thử thất bại.")
        return []

async def stage2_llm_cleaning(df: pd.DataFrame, text_col="Nội dung đánh giá", batch_size: int = 10, max_concurrent: int = 15, checkpoint_file="checkpoint_stage2.xlsx"):
    print(f"[*] Giai đoạn 2: Xử lý LLM với cơ chế Checkpoint...")
    
    # 1. Khởi tạo danh sách gốc
    reviews_list = [{"id": i, "text": text} for i, text in enumerate(df[text_col].tolist())]
    
    # 2. Đọc Checkpoint (Nếu đã từng chạy dở dang)
    processed_ids = set()
    checkpoint_data = []
    
    if os.path.exists(checkpoint_file):
        try:
            df_ckpt = pd.read_excel(checkpoint_file)
            processed_ids = set(df_ckpt['id'].tolist())
            checkpoint_data = df_ckpt.to_dict('records')
            print(f"[+] Tìm thấy file Checkpoint! Đã khôi phục {len(processed_ids)} dòng đã xử lý thành công.")
        except Exception as e:
            print(f"[!] Lỗi đọc file checkpoint (Có thể file bị hỏng): {e}")

    # 3. Lọc ra những câu CHƯA được xử lý
    remaining_reviews = [r for r in reviews_list if r["id"] not in processed_ids]
    
    if not remaining_reviews:
        print("[+] Tuyệt vời! Toàn bộ dữ liệu đã được xử lý xong từ trước.")
        return pd.DataFrame(checkpoint_data)

    # 4. Chia lô cho phần dữ liệu CÒN LẠI
    batches = [remaining_reviews[i:i + batch_size] for i in range(0, len(remaining_reviews), batch_size)]
    print(f"[*] Cần xử lý {len(remaining_reviews)} câu còn lại. Đã chia thành {len(batches)} lô (Mỗi lô max {batch_size} câu).")
    
    semaphore = asyncio.Semaphore(max_concurrent)
    
    # --- CƠ CHẾ CHECKPOINT ---
    checkpoint_interval = 10 # Cứ 10 lô (tương đương 100 câu) thì lưu file 1 lần
    
    print("\nBắt đầu gửi request tới API...")
    with tqdm(total=len(batches), desc="Tiến độ xử lý lô", unit=" lô") as pbar:
        # Vòng lặp duyệt qua từng cụm 10 lô
        for i in range(0, len(batches), checkpoint_interval):
            chunk_batches = batches[i:i + checkpoint_interval]
            tasks = [process_batch_async(batch, semaphore) for batch in chunk_batches]
            
            # Đợi xử lý xong cụm hiện tại
            chunk_results = await asyncio.gather(*tasks)
            pbar.update(len(chunk_batches))
            
            # Gộp kết quả của cụm này vào dữ liệu tổng
            for batch_res in chunk_results:
                for item in batch_res:
                    # Lấy lại đánh giá gốc từ remaining_reviews bằng id
                    orig_text = next((r["text"] for r in remaining_reviews if r["id"] == item["id"]), "")
                    
                    checkpoint_data.append({
                        "id": item["id"],
                        "Đánh giá gốc": orig_text,
                        "is_valid_fashion_review": item.get("is_valid_fashion_review", False),
                        "Đánh giá LLM chuẩn hóa": item.get("cleaned_text", "")
                    })
                    
            # LƯU FILE CHECKPOINT
            try:
                temp_df = pd.DataFrame(checkpoint_data)
                temp_df.to_excel(checkpoint_file, index=False, engine='openpyxl')
            except Exception as e:
                tqdm.write(f"[!] Lỗi khi ghi file checkpoint: {e}")
                
            # Nghỉ ngơi 1 giây để tránh bị API tính là spam requests
            await asyncio.sleep(1)

    # 5. Lọc kết quả cuối cùng (Chỉ lấy các câu hợp lệ is_valid_fashion_review == True)
    df_final = pd.DataFrame(checkpoint_data)
    df_valid_only = df_final[df_final['is_valid_fashion_review'] == True].copy()
    
    # Cắt bỏ các cột dư thừa không cần thiết cho output cuối
    if not df_valid_only.empty:
        df_valid_only = df_valid_only[['Đánh giá gốc', 'Đánh giá LLM chuẩn hóa']]
        
    return df_valid_only

# --- 6. HÀM CHẠY CHÍNH TỔNG HỢP ---
async def main_pipeline(input_file: str, output_file: str):
    try:
        if input_file.endswith('.csv'):
            df_raw = pd.read_csv(input_file)
        else:
            df_raw = pd.read_excel(input_file)
    except Exception as e:
        print(f"[X] Lỗi khi đọc file: {e}")
        return

    # TỰ ĐỘNG LẤY TÊN CỘT: 
    # Cố gắng tìm 'Nội dung đánh giá' hoặc 'Nội dung review' (như trong Jupyter Notebook của bạn)
    text_column_name = "Nội dung đánh giá"
    if text_column_name not in df_raw.columns:
        if "Nội dung review" in df_raw.columns:
            text_column_name = "Nội dung review"
        else:
            print(f"[X] Không tìm thấy cột chứa đánh giá trong dữ liệu. Các cột hiện có: {list(df_raw.columns)}")
            return
        
    df_stage1 = stage1_basic_cleaning(df_raw, text_col=text_column_name)
    
    if df_stage1.empty:
        print("[!] Không còn dữ liệu nào sau Giai đoạn 1.")
        return

    # Khai báo file checkpoint tạm thời
    CHECKPOINT_FILE = "temp_checkpoint_stage2.xlsx"
    
    # Chạy Giai đoạn 2 (có truyền tên file checkpoint)
    df_final = await stage2_llm_cleaning(df_stage1, text_col=text_column_name, checkpoint_file=CHECKPOINT_FILE)
    
    # Lưu kết quả
    if not df_final.empty:
        df_final.to_excel(output_file, index=False, engine='openpyxl')
        print(f"\n✅ HOÀN TẤT PIPELINE!")
        print(f"Số lượng đánh giá sạch thu được: {len(df_final)}")
        print(f"File kết quả đã được lưu tại: {output_file}")
        
        # (Tuỳ chọn) Xoá file checkpoint đi nếu đã xuất thành công file cuối
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            print("Đã xoá file checkpoint tạm thời.")
    else:
        print("\n❌ Không có dữ liệu hợp lệ nào vượt qua được Giai đoạn 2.")

# --- KHỞI ĐỘNG PIPELINE ---
if __name__ == "__main__":
    INPUT_FILE = "raw_fashion_reviews.xlsx" 
    OUTPUT_FILE = "cleaned_fashion_reviews.xlsx"
    
    # Khởi chạy Pipeline
    await main_pipeline(INPUT_FILE, OUTPUT_FILE)

[*] Giai đoạn 1: Bắt đầu dọn dẹp cơ bản với 184 dòng dữ liệu...
[*] Giai đoạn 1 hoàn tất. Còn lại 156 dòng hợp lệ.
[*] Giai đoạn 2: Xử lý LLM với cơ chế Checkpoint...
[*] Cần xử lý 156 câu còn lại. Đã chia thành 16 lô (Mỗi lô max 10 câu).

Bắt đầu gửi request tới API...


Tiến độ xử lý lô:   0%|                                                                        | 0/16 [00:40<?, ? lô/s]

[!] Lỗi giải mã JSON ở lô ID 40 (Thử lại 1/3): Unterminated string starting at: line 1 column 973 (char 972)


Tiến độ xử lý lô:   0%|                                                                        | 0/16 [00:43<?, ? lô/s]

[!] Lỗi giải mã JSON ở lô ID 20 (Thử lại 1/3): Unterminated string starting at: line 1 column 930 (char 929)


Tiến độ xử lý lô: 100%|███████████████████████████████████████████████████████████████| 16/16 [01:27<00:00,  5.46s/ lô]


✅ HOÀN TẤT PIPELINE!
Số lượng đánh giá sạch thu được: 149
File kết quả đã được lưu tại: cleaned_fashion_reviews.xlsx
Đã xoá file checkpoint tạm thời.
